# Demo: Event Summarization using SmolVLM

This notebook provides a demonstration for the second task of the assignment. Given the strong performance of vision-language models across a variety of tasks, we use an efficient, lightweight model: [SmolVLM 2](https://huggingface.co/blog/smolvlm2)
The goal of this notebook is to guide you on how to: design prompts for event detection, process model outputs, and evaluate the quality of detected events.
You are expected to build upon this baseline and critically evaluate your chosen approaches.

In [ ]:
!pip install num2words

In [ ]:
!pip install av


SmolVLM2 is a lightweight vision-language model designed for efficient video understanding with relatively few parameters. The material presented here is largely based on the original documentation and is intended to provide a starting point for developing your own approach.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
import cv2
from PIL import Image
import random
model_path = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
).to("cuda")

Since SmolVLM2 is a vision-language model (VLM), you will need to design an appropriate prompt for the task. The prompt can be flexible and should be adapted based on the type of output you want (e.g., event lists, descriptions, or timestamps).

Vision-language models process visual inputs (such as frames or videos) together with text by encoding them into a shared representation space. The exact input format may vary across models.

In our case, the HuggingFace implementation handles most of the input preprocessing (e.g., frame sampling and formatting), allowing you to focus primarily on prompt design and output parsing.

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "video", "path": "video_1.mp4"},
            {"type": "text", "text": "Describe the most relevant events in the video, list each event sequentially, using a numbered format. Describe it in terms of the actions different persons take"}
        ]
    },
]

inputs = processor.apply_chat_template(
    messages,
    num_frames = 15,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16) # Considering the size of the model, and the number of frames we are sampling are few. This is ultimately a choice that you must make
generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=128)
generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)

print(generated_texts[0])

The detected events are often highly descriptive, but they do not always capture the most important or salient moments in the video.

To address this, you may explore alternative strategies. One approach is to use a language model (LLM) to process frame-level descriptions generated from the video.

For example, you can experiment with randomly sampling frames from the video, generating descriptions for each frame using a vision-language model (VLM), and then aggregating these descriptions into a structured set of events using an LLM.


In [ ]:
def sample_frames(video_path, num_frames):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        raise ValueError("Video has no frames.")

    frame_indices = sorted(random.sample(range(total_frames), min(num_frames, total_frames)))

    frames = []
    current_idx = 0
    target_idx_set = set(frame_indices)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        if current_idx in target_idx_set:
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame))

        current_idx += 1

    cap.release()
    return frames

In [ ]:
messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": frames[0]},
            {"type": "image", "url": frames[1]},
            {"type": "image", "url": frames[2]},
            {"type": "image", "url": frames[3]},
            {"type": "text", "text": "Describe the most relevant events in the video, list each event sequentially, using a numbered format. Describe it in terms of the actions different persons take"}
        ]
    },
]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

You may improve event detection by splitting the video into smaller temporal segments and prompting the VLM on each segment separately. The resulting partial event descriptions can then be combined into a single, coherent list using an LLM.

Crucially, you must define an evaluation scheme for event coverage, for example:
-How many annotated events are correctly detected?
-Which events are missed or incorrectly predicted?
-How does performance change with different prompting or segmentation strategies?

In [ ]:
generated_ids = model.generate(**inputs, do_sample=False, max_new_tokens=256)

generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)

print(generated_texts[0])

The generated outputs may be quite descriptive. While this can introduce noise, it also provides an opportunity to infer potential events through reasoning over these descriptions.

Therefore, one possible strategy is to first generate scene-level descriptions using a vision-language model (VLM), and then further process these descriptions using a language model (LLM) to extract structured events.

The choice of strategy is left to you. You are expected to design and implement an approach that produces meaningful event representations from the video.